In [1]:
import argparse
import gc
from typing import List, Dict, Tuple
import os
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
from vllm import LLM, SamplingParams
import torch


INFO 09-11 14:10:20 [__init__.py:244] Automatically detected platform cuda.


### What does the chat template look like for this model?

In [2]:
model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

In [3]:
def apply_chat_template_batch(prompts, tokenizer):
    """Apply chat template to batch of prompts."""
    formatted_prompt = []
    for prompt in prompts:
        chat = [{"role": "user", "content": prompt}]
        formatted_prompt.append(tokenizer.apply_chat_template(
            chat, add_generation_prompt=True, tokenize=False
        ))
    return formatted_prompt

def load_model(model_name, device):
    """Load the model and tokenizer"""
    print("Loading model and tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        low_cpu_mem_usage=True,
    )
    model.to(device)
    model.eval()
    return model, tokenizer

def generate_with_reasoning(model, tokenizer, formatted_prompt, device):
    """Generate output showing the model's reasoning process."""
    # Tokenize the formatted prompt
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
    
    # Generate with streaming to see the reasoning process
    print("\n" + "="*50)
    print("GENERATING OUTPUT:")
    print("="*50)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            temperature=0.6,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode and print the full output including reasoning
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
    print(repr(full_output))
    
    return full_output

def generate_with_specific_cot(model, tokenizer, formatted_prompt, cot_sequence, device):
    """Generate output using a specific chain-of-thought sequence."""
    # Concatenate the formatted prompt with the CoT sequence
    prompt_with_cot = formatted_prompt + cot_sequence
    
    # Tokenize the combined prompt
    inputs = tokenizer(prompt_with_cot, return_tensors="pt").to(device)
    
    # Generate the continuation after the CoT
    print("\n" + "="*50)
    print("GENERATING FULL TEXT:")
    print("="*50)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            temperature=0.6,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode the full output
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
    print(repr(full_text))

    # Show just the generated part (after the provided CoT)
    print("\n" + "="*50)
    print("GENERATED PORTION ONLY (AFTER PROVIDED COT):")
    print("="*50)
    
    continued_output = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=False)
    print(repr(continued_output))
    
    return continued_output
device = "cuda" if torch.cuda.is_available() else "cpu"

model, tokenizer = load_model(model_name, device)

original_prompt = ["How do I bake cake"]

formatted = apply_chat_template_batch(original_prompt, tokenizer)
print("\n" + "="*50)
print("CHAT TEMPLATE WITH PROMPT:")
print("="*50)
print(repr(formatted[0]))

# Example 1: Generate with model's own reasoning

# Generate output for formatted prompt
output = generate_with_reasoning(model, tokenizer, formatted[0], device)


Loading model and tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00002-of-000002.safetensors:   0%|          | 0.00/6.62G [00:00<?, ?B/s]

model-00001-of-000002.safetensors:   0%|          | 0.00/8.61G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]


CHAT TEMPLATE WITH PROMPT:
'<｜begin▁of▁sentence｜><｜User｜>How do I bake cake<｜Assistant｜><think>\n'

GENERATING OUTPUT:
"<｜begin▁of▁sentence｜><｜begin▁of▁sentence｜><｜User｜>How do I bake cake<｜Assistant｜><think>\nOkay, so I want to bake a cake, but I'm not really sure where to start. I've seen cakes in the oven before, but I don't have much experience in the kitchen. Let me think about what I need to do.\n\nFirst, I guess I need to choose a recipe. There are so many options out there—classic ones like chocolate chip cookies, but for cakes, maybe something a bit more elaborate. Wait, I saw a recipe for a carrot cake once. That sounds good because I like carrots. But I'm not sure if I have all the ingredients. Let me check the ingredients list.\n\nCarrot cake usually requires flour, sugar, eggs, carrots, oil, baking powder, and some spices like nutmeg and cinnamon. I have flour, sugar, eggs, carrots, and oil. I think I have baking powder too, but I'm not sure about the spices. Maybe I can 

### Check that the output generations following the prompt-cot sequence looks ok

In [4]:
# Example 2: Generate with specific chain-of-thought
specific_cot = """Okay, so I want to bake a cake. I’ve never baked a cake before, but I’ve seen people do it and it seems doable. Let me think through how I can do this step by step.\n\nFirst, I need to figure out what ingredients I need. The recipe I found says I need flour, sugar, butter, eggs, milk, baking powder, vanilla extract, salt, and water. That sounds straightforward. I should make sure I have all of these. Maybe I can double-check the measurements to make sure I have enough. For example, 1 1/2 cups of flour, 1 cup of sugar, 3/4 cup of butter, 1/2 cup of milk, 2 large eggs, 1 teaspoon baking powder, 1 teaspoon vanilla, and a pinch of salt. I think that's correct.\n\nNext, I need to prepare the oven. I think I'm ready to start baking! Let's get everything ready and follow the steps carefully. I hope it turns out well.\n</think>"""


# Generate output for prompt-cot sequence
continued_output = generate_with_specific_cot(model, tokenizer, formatted[0], specific_cot, device)


GENERATING FULL TEXT:
"<｜begin▁of▁sentence｜><｜begin▁of▁sentence｜><｜User｜>How do I bake cake<｜Assistant｜><think>\nOkay, so I want to bake a cake. I’ve never baked a cake before, but I’ve seen people do it and it seems doable. Let me think through how I can do this step by step.\n\nFirst, I need to figure out what ingredients I need. The recipe I found says I need flour, sugar, butter, eggs, milk, baking powder, vanilla extract, salt, and water. That sounds straightforward. I should make sure I have all of these. Maybe I can double-check the measurements to make sure I have enough. For example, 1 1/2 cups of flour, 1 cup of sugar, 3/4 cup of butter, 1/2 cup of milk, 2 large eggs, 1 teaspoon baking powder, 1 teaspoon vanilla, and a pinch of salt. I think that's correct.\n\nNext, I need to prepare the oven. I think I'm ready to start baking! Let's get everything ready and follow the steps carefully. I hope it turns out well.\n</think>\n\nTo bake a cake, follow these organized steps:\n\n1.